In [0]:
%pip install databricks-sdk==0.103.0 

In [0]:
%restart_python

In [0]:
%run ./Classroom-Setup-Common-Python

In [0]:
#####################
# CREATE USER OR SET USER CATALOG
#####################
my_catalog = build_user_catalog(catalog_forced=None)

In [0]:
from pathlib import Path
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.dashboards import Dashboard


# Placeholder strings present in the exported template (developer)
ORIGINAL_CATALOG = "labuser_peter_styliadis"
ORIGINAL_SCHEMA = "aibi_dashboards"
ORIGINAL_LAB_SCHEMA = "aibi_dashboards_taxi"

In [0]:

def load_and_rewrite_template(
    template_path: str, 
    ORIGINAL_CATALOG: str,
    ORIGINAL_SCHEMA: str,
    catalog: str, 
    schema: str
) -> str:
    """Read the .lvdash.json template and substitute the catalog and schema.

    Every string occurrence of ORIGINAL_CATALOG is replaced with `catalog`,
    and every string occurrence of ORIGINAL_SCHEMA is replaced with `schema`.
    This covers dataset.catalog / dataset.schema fields, asset_name fully
    qualified references, and any embedded volume URLs.

    Returns:
        The rewritten dashboard JSON as a string.
    """
    text = Path(template_path).read_text(encoding="utf-8")
    text = text.replace(ORIGINAL_CATALOG, catalog)
    text = text.replace(ORIGINAL_SCHEMA, schema)
    return text

In [0]:
DEFAULT_WAREHOUSE_NAME = "shared_warehouse"
FREE_EDITION_WAREHOUSE_NAME = "Serverless Starter Warehouse"


def find_warehouse_id_by_name(
    warehouse_name: str = "shared_warehouse",
    FREE_EDITION_WAREHOUSE_NAME = "Serverless Starter Warehouse"
) -> str:
    """Resolve a SQL warehouse display name to its id.

    Tries `warehouse_name` first. If no warehouse with that name exists, falls
    back to `Serverless Starter Warehouse` (the default in Databricks Free
    Edition). Defaults to `shared_warehouse` if no name is provided.

    Raises:
        ValueError: if neither the provided name nor the free-edition fallback
            resolves to exactly one warehouse.
    """
    w = WorkspaceClient()
    warehouses = list(w.warehouses.list())

    candidates = [warehouse_name]
    if warehouse_name != FREE_EDITION_WAREHOUSE_NAME:
        candidates.append(FREE_EDITION_WAREHOUSE_NAME)

    for candidate in candidates:
        matches = [wh for wh in warehouses if wh.name == candidate]
        if len(matches) == 1:
            return matches[0].id
        if len(matches) > 1:
            raise ValueError(
                f"Multiple SQL warehouses are named {candidate!r}. "
                f"Rename one in your workspace, or pass a unique warehouse id directly."
            )

    available = ", ".join(sorted(wh.name for wh in warehouses)) or "(none)"
    raise ValueError(
        f"Could not find a SQL warehouse named {warehouse_name!r} or "
        f"{FREE_EDITION_WAREHOUSE_NAME!r} in your workspace. "
        f"You must provide a warehouse name that is available in your workspace. "
        f"Available warehouses: {available}"
    )


In [0]:

def create_solution_dashboard(
    catalog: str,
    schema: str,
    display_name: str,
    parent_path: str,
    serialized_json,
    warehouse_id: str = None
) -> Dashboard:
    """Rewrite the template and create the dashboard in the workspace.

    Args:
        template_path: Path to the .lvdash.json template (sibling file works as `./<name>`).
        catalog: Catalog name to substitute into the template.
        schema: Schema name to substitute into the template.
        display_name: Dashboard name shown in the workspace listing.
        parent_path: Workspace path under which to create the dashboard.
        warehouse_name: Optional SQL warehouse display name to bind the dashboard to.
            The name is resolved to an id via the SDK before creating.

    Returns:
        The created Dashboard object (includes dashboard_id and path).
    """
    w = WorkspaceClient()
    return w.lakeview.create(
        dashboard=Dashboard(
            display_name=display_name,
            parent_path=parent_path,
            serialized_dashboard=serialized_json,
            warehouse_id=warehouse_id,
        )
    )
